# BERT_sentiment_analysis_wikipedia

Este notebook implementa um **BERT pequeno do zero** em PyTorch e o treina com dados reais em português.

A ideia é separar duas etapas:

1. **Pré-treinamento auto-supervisionado** na Wikipedia em português, com os objetivos do BERT original:
   - **Masked Language Modeling (MLM)**: prever tokens ocultos por `[MASK]`.
   - **Next Sentence Prediction (NSP)**: decidir se uma sentença vem logo depois da outra.
2. **Fine-tuning downstream** reaproveitando o mesmo encoder BERT em três tarefas:
   - análise de sentimentos;
   - NER, reconhecimento de entidades nomeadas;
   - Q&A extrativo, pergunta e resposta por span.

> Esta é uma implementação didática. Ela usa arquitetura, tokenizer e pesos treinados do zero, mas em escala reduzida para caber em um notebook. Um BERT-base real exige corpus, GPU/TPU e tempo de treino muito maiores.


## Referências

- Devlin et al. **BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding**. 2018.
- Jurafsky e Martin. **Speech and Language Processing**, capítulo 10, seção sobre masked language models.

Conceitos centrais usados aqui:

- O BERT é um **encoder Transformer bidirecional**.
- A entrada combina embeddings de **token**, **posição** e **segmento**.
- O token `[CLS]` serve como representação agregada para tarefas de classificação.
- Em NER, classificamos cada posição da sequência e ignoramos subpalavras ou tokens especiais na loss.
- Em Q&A extrativo, o modelo prediz duas distribuições: início e fim da resposta dentro do contexto.



## Setup básico para Google Colab

Este notebook foi adaptado para ser executado no **Google Colab**, preferencialmente com GPU habilitada em `Ambiente de execução > Alterar tipo de ambiente de execução > GPU`.

O pré-treinamento longo usado neste trabalho foi realizado localmente em um **iMac M4** usando o backend **MPS** do PyTorch. As configurações do iMac foram mantidas comentadas na célula de hiperparâmetros como referência de reprodutibilidade.

No Colab, a configuração padrão abaixo é mais conservadora: ela prioriza carregar o tokenizer/checkpoint já salvos e executar inferência ou fine-tuning leve, sem recomeçar o treinamento longo do zero.


In [ ]:

# Google Colab: setup básico de dependências.
# O Colab normalmente já vem com PyTorch instalado; por isso instalamos apenas pacotes extras.
import os
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("Executando no Colab:", IN_COLAB)

if IN_COLAB:
    !{sys.executable} -m pip install -q datasets tokenizers seqeval scikit-learn pandas numpy matplotlib tqdm
else:
    print("Fora do Colab: se necessário, instale com `pip install -r requirements.txt`.")


In [ ]:

import html
import math
import os
import random
import re
import time
import unicodedata
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from seqeval.metrics import classification_report, f1_score as seqeval_f1
from sklearn.metrics import accuracy_score, f1_score
from tokenizers import Tokenizer
from tokenizers.decoders import WordPiece as WordPieceDecoder
from tokenizers.models import WordPiece
from tokenizers.normalizers import BertNormalizer
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import WordPieceTrainer
from torch.utils.data import DataLoader, Dataset, IterableDataset
from tqdm.auto import tqdm

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

PROJECT_NAME = "BERT_sentiment_analysis_wikipedia"

# Para usar Google Drive no Colab, mude para True e copie os artefatos para a pasta criada.
USE_GOOGLE_DRIVE = False

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
elif IN_COLAB:
    BASE_DIR = Path("/content")
else:
    BASE_DIR = Path(".")

OUTPUT_DIR = BASE_DIR / f"{PROJECT_NAME}_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# No Colab, a GPU CUDA deve ter prioridade. Em execução local no iMac, usamos MPS se existir.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    ACCELERATOR = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    ACCELERATOR = "mps"
else:
    DEVICE = torch.device("cpu")
    ACCELERATOR = "cpu"

USE_AMP = ACCELERATOR == "cuda"
if ACCELERATOR == "cuda":
    torch.backends.cudnn.benchmark = True

print("Colab:", IN_COLAB)
print("Acelerador:", ACCELERATOR)
print("Device:", DEVICE)
print("AMP:", USE_AMP)
print("Diretório de artefatos:", OUTPUT_DIR)

if ACCELERATOR == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("Memória total GPU (GB):", round(props.total_memory / 1024**3, 2))
elif ACCELERATOR == "mps":
    print("MPS disponível: usando GPU integrada Apple Silicon via PyTorch.")



## Configuração para Google Colab

A célula abaixo usa um preset básico para Colab com GPU CUDA. Ele é intencionalmente menor que o treino longo realizado no iMac M4, porque o objetivo ao subir o notebook no Colab é conseguir executar o pipeline, carregar artefatos e fazer inferência/fine-tuning leve sem estourar memória.

As configurações usadas no iMac M4 para o treinamento longo foram mantidas comentadas dentro da própria célula, como registro do experimento realizado localmente.


In [ ]:

import math
import torch

# Esta célula pode ser executada sozinha após reiniciar o kernel.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    ACCELERATOR = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    ACCELERATOR = "mps"
else:
    DEVICE = torch.device("cpu")
    ACCELERATOR = "cpu"

USE_AMP = ACCELERATOR == "cuda"
if ACCELERATOR == "cuda":
    torch.backends.cudnn.benchmark = True

print("Colab:", IN_COLAB)
print("Acelerador configurado:", ACCELERATOR)
print("Device:", DEVICE)
print("AMP:", USE_AMP)

# A Wikipedia portuguesa 20231101 possui aproximadamente 1.112.246 artigos.
WIKIPEDIA_PT_TOTAL_DOCS = 1_112_246

if ACCELERATOR == "cuda":
    # Preset básico para Google Colab com GPU T4/L4/A100.
    # Mantém a mesma arquitetura do checkpoint treinado no iMac, mas reduz o volume de treino.
    VOCAB_SIZE = 20_000
    MAX_LEN = 128
    MAX_QUESTION_LEN = 32

    HIDDEN_SIZE = 256
    NUM_LAYERS = 4
    NUM_HEADS = 4
    INTERMEDIATE_SIZE = 1024
    DROPOUT = 0.1

    MAX_WIKI_DOCS = 10_000
    TOKENIZER_TRAIN_WIKI_DOCS = 10_000
    MAX_SENTENCES_PER_DOC = 8
    MAX_EXTRA_TEXTS_PER_SOURCE = 1_000
    PRETRAIN_NEGATIVE_BUFFER_SENTENCES = 4_000

    MLM_PROBABILITY = 0.15
    PRETRAIN_STEPS = 2_000
    PRETRAIN_BATCH_SIZE = 16
    PRETRAIN_LR = 3e-4

    DOWNSTREAM_EPOCHS = 1
    DOWNSTREAM_LR = 5e-5
    DOWNSTREAM_BATCH_SIZE = 16

    SENTIMENT_PER_CLASS_TRAIN = 1_000
    SENTIMENT_PER_CLASS_VAL = 200
    NER_TRAIN_SIZE = 1_000
    NER_VAL_SIZE = 240
    QA_TRAIN_SIZE = 350
    QA_VAL_SIZE = 80

else:
    # Fallback leve para CPU ou execução local sem CUDA.
    VOCAB_SIZE = 12_000
    MAX_LEN = 96
    MAX_QUESTION_LEN = 24

    HIDDEN_SIZE = 192
    NUM_LAYERS = 3
    NUM_HEADS = 3
    INTERMEDIATE_SIZE = 768
    DROPOUT = 0.1

    MAX_WIKI_DOCS = 1_000
    TOKENIZER_TRAIN_WIKI_DOCS = 1_000
    MAX_SENTENCES_PER_DOC = 6
    MAX_EXTRA_TEXTS_PER_SOURCE = 300
    PRETRAIN_NEGATIVE_BUFFER_SENTENCES = 800

    MLM_PROBABILITY = 0.15
    PRETRAIN_STEPS = 300
    PRETRAIN_BATCH_SIZE = 4
    PRETRAIN_LR = 3e-4

    DOWNSTREAM_EPOCHS = 1
    DOWNSTREAM_LR = 5e-5
    DOWNSTREAM_BATCH_SIZE = 4

    SENTIMENT_PER_CLASS_TRAIN = 120
    SENTIMENT_PER_CLASS_VAL = 40
    NER_TRAIN_SIZE = 160
    NER_VAL_SIZE = 60
    QA_TRAIN_SIZE = 80
    QA_VAL_SIZE = 30

# -------------------------------------------------------------------
# Configuração usada no iMac M4 para o treinamento longo já realizado
# -------------------------------------------------------------------
# O treinamento principal do checkpoint usado no trabalho foi feito localmente
# no iMac M4 com backend MPS. Mantemos o preset abaixo comentado apenas como
# referência; no Colab, use o bloco CUDA acima.
#
# VOCAB_SIZE = 20_000
# MAX_LEN = 128
# MAX_QUESTION_LEN = 32
# HIDDEN_SIZE = 256
# NUM_LAYERS = 4
# NUM_HEADS = 4
# INTERMEDIATE_SIZE = 1024
# DROPOUT = 0.1
# MAX_WIKI_DOCS = WIKIPEDIA_PT_TOTAL_DOCS
# TOKENIZER_TRAIN_WIKI_DOCS = 80_000
# MAX_SENTENCES_PER_DOC = 8
# MAX_EXTRA_TEXTS_PER_SOURCE = 2_000
# PRETRAIN_NEGATIVE_BUFFER_SENTENCES = 20_000
# MLM_PROBABILITY = 0.15
# PRETRAIN_STEPS = 220_000
# PRETRAIN_BATCH_SIZE = 32
# PRETRAIN_LR = 2e-4
# DOWNSTREAM_EPOCHS = 1
# DOWNSTREAM_LR = 5e-5
# DOWNSTREAM_BATCH_SIZE = 8
# SENTIMENT_PER_CLASS_TRAIN = 1_000
# SENTIMENT_PER_CLASS_VAL = 200
# NER_TRAIN_SIZE = 1_000
# NER_VAL_SIZE = 240
# QA_TRAIN_SIZE = 350
# QA_VAL_SIZE = 80

PRETRAIN_WARMUP_STEPS = min(10_000, max(50, PRETRAIN_STEPS // 10))
CHECKPOINT_EVERY_STEPS = 500 if IN_COLAB else 10_000

# Workers > 0 podem duplicar streams do Hugging Face e aumentar RAM em notebooks.
NUM_DATALOADER_WORKERS = 0
PIN_MEMORY = ACCELERATOR == "cuda"
DL_KWARGS = {"num_workers": NUM_DATALOADER_WORKERS, "pin_memory": PIN_MEMORY}

print({
    "IN_COLAB": IN_COLAB,
    "ACCELERATOR": ACCELERATOR,
    "DEVICE": str(DEVICE),
    "MAX_WIKI_DOCS": MAX_WIKI_DOCS,
    "TOKENIZER_TRAIN_WIKI_DOCS": TOKENIZER_TRAIN_WIKI_DOCS,
    "VOCAB_SIZE": VOCAB_SIZE,
    "MAX_LEN": MAX_LEN,
    "HIDDEN_SIZE": HIDDEN_SIZE,
    "NUM_LAYERS": NUM_LAYERS,
    "NUM_HEADS": NUM_HEADS,
    "INTERMEDIATE_SIZE": INTERMEDIATE_SIZE,
    "PRETRAIN_BATCH_SIZE": PRETRAIN_BATCH_SIZE,
    "PRETRAIN_STEPS": PRETRAIN_STEPS,
    "PRETRAIN_WARMUP_STEPS": PRETRAIN_WARMUP_STEPS,
    "CHECKPOINT_EVERY_STEPS": CHECKPOINT_EVERY_STEPS,
    "DOWNSTREAM_BATCH_SIZE": DOWNSTREAM_BATCH_SIZE,
})


## 1. Corpus real para pré-treinamento

O pré-treinamento principal usa `wikimedia/wikipedia`, configuração `20231101.pt`, em streaming.

Atenção: não guardamos 50% da Wikipedia em uma lista Python. Isso foi o que levou ao crescimento da RAM do Colab. Em vez disso, usamos geradores/`IterableDataset` e um buffer limitado para NSP negativo.


In [66]:
def dividir_sentencas(texto: str, min_chars: int = 30) -> List[str]:
    texto = html.unescape(texto or "")
    texto = re.sub(r"\s+", " ", texto).strip()
    partes = re.split(r"(?<=[.!?])\s+", texto)
    return [p.strip() for p in partes if len(p.strip()) >= min_chars]


def iter_documentos_wikipedia(max_docs: int):
    ds = load_dataset(
        "wikimedia/wikipedia",
        "20231101.pt",
        split="train",
        streaming=True,
    )
    n_docs = 0
    for row in ds:
        sentencas = dividir_sentencas(row.get("text", ""))[:MAX_SENTENCES_PER_DOC]
        if len(sentencas) >= 2:
            yield sentencas
            n_docs += 1
        if n_docs >= max_docs:
            break


def coletar_preview_wikipedia(max_docs: int = 3) -> List[List[str]]:
    return list(iter_documentos_wikipedia(max_docs))


def coletar_textos_downstream_treino(max_por_fonte: int = MAX_EXTRA_TEXTS_PER_SOURCE) -> List[str]:
    textos = []
    fontes = [
        ("sentimentos", lambda: load_dataset("jvanz/portuguese_sentiment_analysis", split="train", streaming=True)),
        ("ner", lambda: load_dataset("unimelb-nlp/wikiann", "pt", split="train", streaming=True)),
        ("qa", lambda: load_dataset("nunorc/squad_v1_pt", split="train", streaming=True)),
    ]

    for nome, loader in fontes:
        try:
            ds = loader()
            n = 0
            for row in ds:
                if nome == "sentimentos":
                    candidatos = [row.get("review_text", "")]
                elif nome == "ner":
                    candidatos = [" ".join(row.get("tokens", []))]
                else:
                    candidatos = [row.get("question", ""), row.get("context", "")]

                for texto in candidatos:
                    texto = html.unescape(texto or "").strip()
                    if len(texto) >= 30:
                        textos.append(texto)
                        n += 1
                if n >= max_por_fonte:
                    break
            print(f"Textos extras de treino coletados de {nome}: {n}")
        except Exception as exc:
            print(f"Não foi possível coletar textos extras de {nome}: {exc}")

    return textos


documentos_preview = coletar_preview_wikipedia()
textos_extra_pre_treino = coletar_textos_downstream_treino()

print("Preview de documentos da Wikipedia:", len(documentos_preview))
print("Textos extras de treino:", len(textos_extra_pre_treino))
print("Exemplo:", documentos_preview[0][0][:300])


Repo card metadata block was not found. Setting CardData to empty.


Textos extras de treino coletados de sentimentos: 2000
Textos extras de treino coletados de ner: 2000
Textos extras de treino coletados de qa: 2001
Preview de documentos da Wikipedia: 3
Textos extras de treino: 6001
Exemplo: Astronomia é uma ciência natural que estuda corpos celestes (como estrelas, planetas, cometas, nebulosas, aglomerados de estrelas, galáxias) e fenômenos que se originam fora da atmosfera da Terra (como a radiação cósmica de fundo em micro-ondas).


## 2. Tokenizer WordPiece treinado do zero

O BERT original usa WordPiece. Aqui treinamos um tokenizer próprio no corpus real em português. Os tokens especiais são os mesmos usados no artigo: `[PAD]`, `[UNK]`, `[CLS]`, `[SEP]` e `[MASK]`.


In [67]:
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
PAD, UNK, CLS, SEP, MASK = SPECIAL_TOKENS


tokenizer_path = OUTPUT_DIR / f"{PROJECT_NAME}_tokenizer.json"


def iter_textos_para_tokenizer() -> Iterable[str]:
    for doc in iter_documentos_wikipedia(TOKENIZER_TRAIN_WIKI_DOCS):
        for sentenca in doc:
            yield sentenca
    for texto in textos_extra_pre_treino:
        yield texto


if tokenizer_path.exists():
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    print("Tokenizer carregado de:", tokenizer_path)
else:
    tokenizer = Tokenizer(WordPiece(unk_token=UNK))
    tokenizer.normalizer = BertNormalizer(lowercase=True, strip_accents=False)
    tokenizer.pre_tokenizer = BertPreTokenizer()
    trainer = WordPieceTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=SPECIAL_TOKENS,
        min_frequency=2,
        continuing_subword_prefix="##",
    )
    tokenizer.train_from_iterator(iter_textos_para_tokenizer(), trainer=trainer)
    tokenizer.post_processor = TemplateProcessing(
        single="[CLS] $A [SEP]",
        pair="[CLS] $A [SEP] $B:1 [SEP]:1",
        special_tokens=[(CLS, tokenizer.token_to_id(CLS)), (SEP, tokenizer.token_to_id(SEP))],
    )
    tokenizer.decoder = WordPieceDecoder(prefix="##")
    tokenizer.save(str(tokenizer_path))
    print("Tokenizer salvo em:", tokenizer_path)

print("Tamanho do vocabulário:", tokenizer.get_vocab_size())

PAD_ID = tokenizer.token_to_id(PAD)
UNK_ID = tokenizer.token_to_id(UNK)
CLS_ID = tokenizer.token_to_id(CLS)
SEP_ID = tokenizer.token_to_id(SEP)
MASK_ID = tokenizer.token_to_id(MASK)
SPECIAL_IDS = {PAD_ID, UNK_ID, CLS_ID, SEP_ID, MASK_ID}

exemplo = "O BERT aprende representações profundas de textos em português."
enc = tokenizer.encode(exemplo)
print("Tokens:", enc.tokens)
print("IDs:", enc.ids)


Tokenizer carregado de: BERT_sentiment_analysis_wikipedia_outputs/BERT_sentiment_analysis_wikipedia_tokenizer.json
Tamanho do vocabulário: 20000
Tokens: ['[CLS]', 'o', 'ber', '##t', 'aprend', '##e', 'representações', 'profundas', 'de', 'textos', 'em', 'português', '.', '[SEP]']
IDs: [2, 57, 5328, 2807, 10542, 2804, 16529, 18311, 4131, 9244, 4162, 5190, 18, 3]


## 3. Funções de codificação e masking

A regra de masking segue o artigo do BERT: mascaramos 15% dos tokens candidatos. Desses tokens, 80% viram `[MASK]`, 10% viram um token aleatório e 10% permanecem iguais.


In [68]:
def pad_lista(valores: List[int], max_len: int, pad_value: int) -> List[int]:
    return valores + [pad_value] * max(0, max_len - len(valores))


def codificar_texto(texto: str, max_len: int = MAX_LEN) -> Dict[str, List[int]]:
    enc = tokenizer.encode(texto)
    ids = enc.ids[:max_len]
    type_ids = enc.type_ids[:max_len]
    if ids and ids[-1] != SEP_ID:
        ids[-1] = SEP_ID
    attention_mask = [1] * len(ids)
    return {
        "input_ids": pad_lista(ids, max_len, PAD_ID),
        "token_type_ids": pad_lista(type_ids, max_len, 0),
        "attention_mask": pad_lista(attention_mask, max_len, 0),
    }


def codificar_par(texto_a: str, texto_b: str, max_len: int = MAX_LEN) -> Dict[str, List[int]]:
    enc = tokenizer.encode(texto_a, texto_b)
    ids = enc.ids[:max_len]
    type_ids = enc.type_ids[:max_len]
    if ids and ids[-1] != SEP_ID:
        ids[-1] = SEP_ID
    attention_mask = [1] * len(ids)
    return {
        "input_ids": pad_lista(ids, max_len, PAD_ID),
        "token_type_ids": pad_lista(type_ids, max_len, 0),
        "attention_mask": pad_lista(attention_mask, max_len, 0),
    }


def aplicar_mlm(input_ids: List[int], attention_mask: List[int], mlm_probability: float = MLM_PROBABILITY):
    ids = list(input_ids)
    labels = [-100] * len(ids)
    candidatos = [
        i for i, tok in enumerate(ids)
        if attention_mask[i] == 1 and tok not in SPECIAL_IDS
    ]
    if not candidatos:
        return ids, labels

    n_mascaras = max(1, int(round(len(candidatos) * mlm_probability)))
    random.shuffle(candidatos)
    mascarados = candidatos[:n_mascaras]
    vocab_size = tokenizer.get_vocab_size()

    for pos in mascarados:
        original = ids[pos]
        labels[pos] = original
        p = random.random()
        if p < 0.8:
            ids[pos] = MASK_ID
        elif p < 0.9:
            novo = random.randrange(vocab_size)
            while novo in SPECIAL_IDS:
                novo = random.randrange(vocab_size)
            ids[pos] = novo
        else:
            ids[pos] = original

    return ids, labels


demo = codificar_par("O BERT usa atenção.", "Ele aprende com textos reais.")
masked_ids, mlm_labels = aplicar_mlm(demo["input_ids"], demo["attention_mask"])
print("input_ids shape:", torch.tensor(demo["input_ids"]).shape)
print("Tokens mascarados:", tokenizer.decode(masked_ids))
print("Labels MLM válidos:", sum(x != -100 for x in mlm_labels))


input_ids shape: torch.Size([128])
Tokens mascarados: o bert usa atenção.e com textos reais.
Labels MLM válidos: 2


## 4. Arquitetura BERT em PyTorch

Abaixo está uma implementação compacta do encoder BERT:

- embeddings de palavra, posição e segmento;
- self-attention multi-head bidirecional;
- feed-forward com GELU;
- residual connection e LayerNorm;
- pooler sobre `[CLS]`;
- heads para MLM, NSP e downstream.


In [69]:
@dataclass
class BertConfigDidatico:
    vocab_size: int
    max_len: int = MAX_LEN
    hidden_size: int = HIDDEN_SIZE
    num_layers: int = NUM_LAYERS
    num_heads: int = NUM_HEADS
    intermediate_size: int = INTERMEDIATE_SIZE
    type_vocab_size: int = 2
    dropout: float = DROPOUT


class BertEmbeddings(nn.Module):
    def __init__(self, config: BertConfigDidatico):
        super().__init__()
        self.word_embeddings = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=PAD_ID)
        self.position_embeddings = nn.Embedding(config.max_len, config.hidden_size)
        self.token_type_embeddings = nn.Embedding(config.type_vocab_size, config.hidden_size)
        self.layer_norm = nn.LayerNorm(config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids, token_type_ids):
        batch_size, seq_len = input_ids.shape
        position_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, seq_len)
        x = (
            self.word_embeddings(input_ids)
            + self.position_embeddings(position_ids)
            + self.token_type_embeddings(token_type_ids)
        )
        return self.dropout(self.layer_norm(x))


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, config: BertConfigDidatico):
        super().__init__()
        if config.hidden_size % config.num_heads != 0:
            raise ValueError("hidden_size precisa ser divisível por num_heads")
        self.num_heads = config.num_heads
        self.head_dim = config.hidden_size // config.num_heads
        self.q_proj = nn.Linear(config.hidden_size, config.hidden_size)
        self.k_proj = nn.Linear(config.hidden_size, config.hidden_size)
        self.v_proj = nn.Linear(config.hidden_size, config.hidden_size)
        self.out_proj = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, attention_mask):
        bsz, seq_len, hidden = x.shape
        q = self.q_proj(x).view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        padding_mask = attention_mask[:, None, None, :]
        scores = scores.masked_fill(padding_mask == 0, -1e9)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        context = torch.matmul(attn, v)
        context = context.transpose(1, 2).contiguous().view(bsz, seq_len, hidden)
        return self.out_proj(context)


class FeedForward(nn.Module):
    def __init__(self, config: BertConfigDidatico):
        super().__init__()
        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.fc2 = nn.Linear(config.intermediate_size, config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))


class BertLayer(nn.Module):
    def __init__(self, config: BertConfigDidatico):
        super().__init__()
        self.attention = MultiHeadSelfAttention(config)
        self.norm1 = nn.LayerNorm(config.hidden_size)
        self.ffn = FeedForward(config)
        self.norm2 = nn.LayerNorm(config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, attention_mask):
        x = self.norm1(x + self.dropout(self.attention(x, attention_mask)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


class BertModelDidatico(nn.Module):
    def __init__(self, config: BertConfigDidatico):
        super().__init__()
        self.config = config
        self.embeddings = BertEmbeddings(config)
        self.layers = nn.ModuleList([BertLayer(config) for _ in range(config.num_layers)])
        self.pooler = nn.Linear(config.hidden_size, config.hidden_size)

    def forward(self, input_ids, token_type_ids, attention_mask):
        x = self.embeddings(input_ids, token_type_ids)
        for layer in self.layers:
            x = layer(x, attention_mask)
        pooled = torch.tanh(self.pooler(x[:, 0]))
        return x, pooled


class BertMLMHead(nn.Module):
    def __init__(self, config: BertConfigDidatico, embedding_weight: nn.Parameter):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.layer_norm = nn.LayerNorm(config.hidden_size)
        self.decoder_bias = nn.Parameter(torch.zeros(config.vocab_size))
        self.embedding_weight = embedding_weight

    def forward(self, hidden_states):
        x = self.layer_norm(F.gelu(self.dense(hidden_states)))
        return torch.matmul(x, self.embedding_weight.t()) + self.decoder_bias


class BertForPreTrainingDidatico(nn.Module):
    def __init__(self, config: BertConfigDidatico):
        super().__init__()
        self.bert = BertModelDidatico(config)
        self.mlm_head = BertMLMHead(config, self.bert.embeddings.word_embeddings.weight)
        self.nsp_head = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids, token_type_ids, attention_mask):
        sequence_output, pooled_output = self.bert(input_ids, token_type_ids, attention_mask)
        return self.mlm_head(sequence_output), self.nsp_head(pooled_output)


class BertForSequenceClassificationDidatico(nn.Module):
    def __init__(self, config: BertConfigDidatico, num_labels: int):
        super().__init__()
        self.bert = BertModelDidatico(config)
        self.dropout = nn.Dropout(config.dropout)
        self.classifier = nn.Linear(config.hidden_size, num_labels)

    def forward(self, input_ids, token_type_ids, attention_mask):
        _, pooled = self.bert(input_ids, token_type_ids, attention_mask)
        return self.classifier(self.dropout(pooled))


class BertForTokenClassificationDidatico(nn.Module):
    def __init__(self, config: BertConfigDidatico, num_labels: int):
        super().__init__()
        self.bert = BertModelDidatico(config)
        self.dropout = nn.Dropout(config.dropout)
        self.classifier = nn.Linear(config.hidden_size, num_labels)

    def forward(self, input_ids, token_type_ids, attention_mask):
        sequence_output, _ = self.bert(input_ids, token_type_ids, attention_mask)
        return self.classifier(self.dropout(sequence_output))


class BertForQuestionAnsweringDidatico(nn.Module):
    def __init__(self, config: BertConfigDidatico):
        super().__init__()
        self.bert = BertModelDidatico(config)
        self.qa_outputs = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids, token_type_ids, attention_mask):
        sequence_output, _ = self.bert(input_ids, token_type_ids, attention_mask)
        logits = self.qa_outputs(sequence_output)
        start_logits, end_logits = logits.split(1, dim=-1)
        return start_logits.squeeze(-1), end_logits.squeeze(-1)


bert_config = BertConfigDidatico(vocab_size=tokenizer.get_vocab_size())
print(bert_config)


BertConfigDidatico(vocab_size=20000, max_len=128, hidden_size=256, num_layers=4, num_heads=4, intermediate_size=1024, type_vocab_size=2, dropout=0.1)


## 5. Dataset de pré-treinamento MLM + NSP

Cada amostra contém duas sentenças. Metade dos pares é positiva, isto é, a segunda sentença realmente vem depois da primeira no mesmo artigo. A outra metade recebe uma sentença aleatória.


In [70]:
class BertPretrainingIterableDataset(IterableDataset):
    def __init__(
        self,
        max_docs: int,
        num_examples: int,
        max_len: int = MAX_LEN,
        negative_buffer_size: int = PRETRAIN_NEGATIVE_BUFFER_SENTENCES,
    ):
        self.max_docs = max_docs
        self.num_examples = num_examples
        self.max_len = max_len
        self.negative_buffer_size = negative_buffer_size

    def _adicionar_buffer(self, buffer: List[str], sentencas: List[str]):
        buffer.extend(sentencas)
        if len(buffer) > self.negative_buffer_size:
            del buffer[: len(buffer) - self.negative_buffer_size]

    def __iter__(self):
        yielded = 0
        negative_buffer = []
        while yielded < self.num_examples:
            for doc in iter_documentos_wikipedia(self.max_docs):
                if yielded >= self.num_examples:
                    break
                if len(doc) < 2:
                    continue
                for idx in range(len(doc) - 1):
                    if yielded >= self.num_examples:
                        break
                    sent_a = doc[idx]
                    if negative_buffer and random.random() < 0.5:
                        sent_b = random.choice(negative_buffer)
                        nsp_label = 0
                    else:
                        sent_b = doc[idx + 1]
                        nsp_label = 1

                    encoded = codificar_par(sent_a, sent_b, self.max_len)
                    mlm_input_ids, mlm_labels = aplicar_mlm(encoded["input_ids"], encoded["attention_mask"])
                    yielded += 1
                    yield {
                        "input_ids": torch.tensor(mlm_input_ids, dtype=torch.long),
                        "token_type_ids": torch.tensor(encoded["token_type_ids"], dtype=torch.long),
                        "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
                        "mlm_labels": torch.tensor(mlm_labels, dtype=torch.long),
                        "nsp_labels": torch.tensor(nsp_label, dtype=torch.long),
                    }
                self._adicionar_buffer(negative_buffer, doc)


def mover_batch(batch):
    return {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}


def contexto_autocast():
    if USE_AMP:
        return torch.cuda.amp.autocast(enabled=True)
    return nullcontext()


def limpar_cache_acelerador():
    if ACCELERATOR == "cuda":
        torch.cuda.empty_cache()
    elif ACCELERATOR == "mps" and hasattr(torch, "mps"):
        torch.mps.empty_cache()


def passo_otimizador(optimizer):
    if ACCELERATOR == "xla":
        xm.optimizer_step(optimizer, barrier=True)
        xm.mark_step()
    else:
        optimizer.step()


def salvar_torch(obj, path):
    if ACCELERATOR == "xla":
        xm.save(obj, str(path))
    else:
        torch.save(obj, path)


pretrain_dataset = BertPretrainingIterableDataset(
    max_docs=MAX_WIKI_DOCS,
    num_examples=PRETRAIN_STEPS * PRETRAIN_BATCH_SIZE,
)
pretrain_loader = DataLoader(
    pretrain_dataset,
    batch_size=PRETRAIN_BATCH_SIZE,
    drop_last=True,
    **DL_KWARGS,
)

batch = next(iter(pretrain_loader))
print("input_ids:", batch["input_ids"].shape)
print("token_type_ids:", batch["token_type_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("mlm_labels:", batch["mlm_labels"].shape)
print("nsp_labels:", batch["nsp_labels"].shape)


input_ids: torch.Size([32, 128])
token_type_ids: torch.Size([32, 128])
attention_mask: torch.Size([32, 128])
mlm_labels: torch.Size([32, 128])
nsp_labels: torch.Size([32])


## 6. Pré-treinamento do BERT do zero

A loss total é a soma da loss de MLM e da loss de NSP, como no BERT original. Durante o treino, acompanhamos a acurácia dos tokens mascarados e a acurácia de NSP.


In [71]:
def ajustar_lr_linear_warmup(optimizer, step: int, total_steps: int, warmup_steps: int, base_lr: float):
    if step < warmup_steps:
        scale = max(1e-8, step / max(1, warmup_steps))
    else:
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        scale = max(0.0, 1.0 - progress)
    for group in optimizer.param_groups:
        group["lr"] = base_lr * scale


def metricas_mlm_nsp(mlm_logits, mlm_labels, nsp_logits, nsp_labels):
    with torch.no_grad():
        mask = mlm_labels != -100
        if mask.any():
            mlm_pred = mlm_logits.argmax(dim=-1)
            mlm_acc_t = (mlm_pred[mask] == mlm_labels[mask]).float().mean()
        else:
            mlm_acc_t = torch.tensor(0.0, device=mlm_labels.device)
        nsp_acc_t = (nsp_logits.argmax(dim=-1) == nsp_labels).float().mean()
    return mlm_acc_t, nsp_acc_t


def tensor_para_float(x):
    if torch.is_tensor(x):
        return float(x.detach().cpu())
    return float(x)


def treinar_pretraining(model, loader, steps: int = PRETRAIN_STEPS):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=PRETRAIN_LR, weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP) if USE_AMP else None

    historico = []
    start_time = time.time()
    loader_iter = iter(loader)

    for step in range(1, steps + 1):
        ajustar_lr_linear_warmup(optimizer, step, steps, PRETRAIN_WARMUP_STEPS, PRETRAIN_LR)
        try:
            batch = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            batch = next(loader_iter)
        batch = mover_batch(batch)

        optimizer.zero_grad(set_to_none=True)
        with contexto_autocast():
            mlm_logits, nsp_logits = model(
                batch["input_ids"],
                batch["token_type_ids"],
                batch["attention_mask"],
            )
            mlm_loss = F.cross_entropy(
                mlm_logits.view(-1, model.bert.config.vocab_size),
                batch["mlm_labels"].view(-1),
                ignore_index=-100,
            )
            nsp_loss = F.cross_entropy(nsp_logits, batch["nsp_labels"])
            loss = mlm_loss + nsp_loss

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            passo_otimizador(optimizer)

        # Métricas somente a cada 50 steps para evitar sincronizações caras e acúmulo em TPU/XLA.
        if step == 1 or step % 50 == 0 or step == steps:
            mlm_acc_t, nsp_acc_t = metricas_mlm_nsp(mlm_logits, batch["mlm_labels"], nsp_logits, batch["nsp_labels"])
            registro = {
                "step": step,
                "loss": tensor_para_float(loss),
                "mlm_loss": tensor_para_float(mlm_loss),
                "nsp_loss": tensor_para_float(nsp_loss),
                "mlm_acc": tensor_para_float(mlm_acc_t),
                "nsp_acc": tensor_para_float(nsp_acc_t),
            }
            historico.append(registro)
            elapsed = time.time() - start_time
            print(
                f"step={step:04d}/{steps} "
                f"loss={registro['loss']:.4f} mlm={registro['mlm_loss']:.4f} nsp={registro['nsp_loss']:.4f} "
                f"mlm_acc={registro['mlm_acc']:.3f} nsp_acc={registro['nsp_acc']:.3f} "
                f"tempo={elapsed/60:.1f}min"
            )

        if CHECKPOINT_EVERY_STEPS and step % CHECKPOINT_EVERY_STEPS == 0:
            checkpoint_latest = OUTPUT_DIR / f"{PROJECT_NAME}_pretrain_latest.pt"
            salvar_torch(
                {
                    "config": asdict(model.bert.config),
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "step": step,
                    "historico": historico,
                },
                checkpoint_latest,
            )
            print("Checkpoint parcial salvo em:", checkpoint_latest)

        del batch, mlm_logits, nsp_logits, mlm_loss, nsp_loss, loss
        if ACCELERATOR in {"cuda", "mps"} and step % 100 == 0:
            limpar_cache_acelerador()

    return pd.DataFrame(historico)


if ACCELERATOR in {"cuda", "mps"}:
    limpar_cache_acelerador()

pretrain_model = BertForPreTrainingDidatico(bert_config).to(DEVICE)
print("Parâmetros:", sum(p.numel() for p in pretrain_model.parameters()) / 1e6, "M")

with torch.no_grad():
    demo_batch = mover_batch(next(iter(pretrain_loader)))
    demo_mlm, demo_nsp = pretrain_model(
        demo_batch["input_ids"],
        demo_batch["token_type_ids"],
        demo_batch["attention_mask"],
    )
print("Saída MLM:", demo_mlm.shape)
print("Saída NSP:", demo_nsp.shape)
del demo_batch, demo_mlm, demo_nsp


Parâmetros: 8.465442 M
Saída MLM: torch.Size([32, 128, 20000])
Saída NSP: torch.Size([32, 2])


In [72]:
# Mude para True somente quando quiser continuar/rodar o pré-treinamento.
RUN_PRETRAINING = False

if RUN_PRETRAINING:
    historico_pretrain = treinar_pretraining(pretrain_model, pretrain_loader, PRETRAIN_STEPS)
    display(historico_pretrain.tail())
else:
    print("Pré-treinamento pulado. A próxima célula carrega o último checkpoint salvo.")


Pré-treinamento pulado. A próxima célula carrega o último checkpoint salvo.


In [ ]:

checkpoint_pretrain = OUTPUT_DIR / f"{PROJECT_NAME}_pretrain_latest.pt"


def carregar_checkpoint_pretraining(path=checkpoint_pretrain):
    if not Path(path).exists():
        print("Checkpoint não encontrado:", path)
        print("No Colab, envie a pasta de artefatos ou monte o Google Drive e ajuste OUTPUT_DIR.")
        print("Continuando com o modelo inicializado em memória, sem pesos pré-treinados carregados.")
        return pretrain_model, bert_config, {}, pd.DataFrame()

    checkpoint = torch.load(path, map_location="cpu")
    campos_config = BertConfigDidatico.__dataclass_fields__.keys()
    config_dict = checkpoint.get("config", asdict(bert_config))
    config_filtrada = {k: v for k, v in config_dict.items() if k in campos_config}

    config_carregada = BertConfigDidatico(**config_filtrada)
    modelo = BertForPreTrainingDidatico(config_carregada)
    modelo.load_state_dict(checkpoint["model_state_dict"])
    modelo.to(DEVICE)
    modelo.eval()

    historico = pd.DataFrame(checkpoint.get("historico", []))
    return modelo, config_carregada, checkpoint, historico


pretrain_model, bert_config, checkpoint_carregado, historico_pretrain = carregar_checkpoint_pretraining()

if checkpoint_carregado:
    print("Checkpoint carregado de:", checkpoint_pretrain)
    print("Step salvo:", checkpoint_carregado.get("step", "não informado"))
    print("Parâmetros:", round(sum(p.numel() for p in pretrain_model.parameters()) / 1e6, 2), "M")

    if len(historico_pretrain) > 0:
        display(historico_pretrain.tail())
        historico_pretrain[["loss", "mlm_loss", "nsp_loss", "mlm_acc", "nsp_acc"]].plot(
            figsize=(10, 4),
            title="Histórico disponível no checkpoint",
        )


@torch.no_grad()
def prever_mlm(texto: str, top_k: int = 5):
    """Prediz os tokens marcados com [MASK] usando o modelo carregado."""
    pretrain_model.eval()
    enc = codificar_texto(texto)
    batch = {k: torch.tensor([v], dtype=torch.long, device=DEVICE) for k, v in enc.items()}
    mask_positions = (batch["input_ids"][0] == MASK_ID).nonzero(as_tuple=False).flatten()
    if len(mask_positions) == 0:
        raise ValueError("Inclua pelo menos um token [MASK] no texto.")

    mlm_logits, nsp_logits = pretrain_model(
        batch["input_ids"],
        batch["token_type_ids"],
        batch["attention_mask"],
    )

    for pos in mask_positions.tolist():
        probs = torch.softmax(mlm_logits[0, pos], dim=-1)
        valores, ids = torch.topk(probs, k=top_k)
        candidatos = [(tokenizer.id_to_token(int(i)), float(v)) for i, v in zip(ids.cpu(), valores.cpu())]
        print(f"Posição {pos}:", candidatos)


@torch.no_grad()
def prever_nsp(sentenca_a: str, sentenca_b: str):
    """Estima se a segunda sentença é sequência natural da primeira."""
    pretrain_model.eval()
    enc = codificar_par(sentenca_a, sentenca_b)
    batch = {k: torch.tensor([v], dtype=torch.long, device=DEVICE) for k, v in enc.items()}
    _, nsp_logits = pretrain_model(
        batch["input_ids"],
        batch["token_type_ids"],
        batch["attention_mask"],
    )
    probs = torch.softmax(nsp_logits[0], dim=-1).cpu().numpy()
    print({"nao_proxima": round(float(probs[0]), 4), "proxima": round(float(probs[1]), 4)})


prever_mlm("O Brasil fica na [MASK].")


## 7. Utilitários para fine-tuning

Cada modelo downstream recebe uma cópia dos pesos do encoder pré-treinado. A cabeça específica da tarefa começa aleatória e é treinada junto com o encoder.


In [74]:
def copiar_encoder_pre_treinado(modelo_downstream: nn.Module, modelo_pretrain: BertForPreTrainingDidatico):
    modelo_downstream.bert.load_state_dict(modelo_pretrain.bert.state_dict())


def treinar_por_epocas(nome, model, train_loader, val_fn, loss_fn, epochs=DOWNSTREAM_EPOCHS, lr=DOWNSTREAM_LR):
    model.to(DEVICE)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = max(1, epochs * len(train_loader))
    warmup_steps = max(1, total_steps // 10)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP) if USE_AMP else None
    step = 0

    for epoch in range(1, epochs + 1):
        perdas = []
        for batch in tqdm(train_loader, desc=f"{nome} epoch {epoch}"):
            step += 1
            ajustar_lr_linear_warmup(optimizer, step, total_steps, warmup_steps, lr)
            batch = mover_batch(batch)
            optimizer.zero_grad(set_to_none=True)
            with contexto_autocast():
                loss = loss_fn(model, batch)
            if USE_AMP:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                passo_otimizador(optimizer)
            perdas.append(tensor_para_float(loss))
            del batch, loss
            if ACCELERATOR in {"cuda", "mps"} and step % 100 == 0:
                limpar_cache_acelerador()
        print(f"{nome} epoch={epoch} loss={np.mean(perdas):.4f}")
        val_fn(model)


# Tarefa A: análise de sentimentos

Entrada: texto de review. Saída: classe binária.

O modelo usa a representação do token `[CLS]`, exatamente a adaptação clássica descrita no artigo do BERT para classificação de sequência.


In [75]:
# Ajuste estes valores para mudar quais exemplos entram na validação.
# 0 pega os primeiros exemplos do split validation; 200 pula os 200 primeiros de cada classe.

SENTIMENT_VAL_SKIP_PER_CLASS = 200
SENTIMENT_PREVIEW_INDICES = [0, 1, 2]

# Opcional: para uma validação manual, preencha esta lista e deixe o dataset validation de lado.
# Labels: 0 = negativo, 1 = positivo.
SENTIMENT_VAL_MANUAL_EXAMPLES = [
    # {"text": "O app trava toda hora e não consigo finalizar a compra.", "label": 0},
    # {"text": "A experiência foi ótima e o atendimento resolveu tudo rapidamente.", "label": 1},
]


def coletar_sentimentos(split: str, por_classe: int, skip_por_classe: int = 0, seed: int = SEED) -> List[Dict]:
    ds = load_dataset("jvanz/portuguese_sentiment_analysis", split=split, streaming=True)
    contagem = {0: 0, 1: 0}
    pulados = {0: 0, 1: 0}
    exemplos = []
    for row in ds:
        label = int(row["polarity"])
        if label not in contagem:
            continue
        texto = html.unescape(row.get("review_text", "")).strip()
        if not texto:
            continue
        if pulados[label] < skip_por_classe:
            pulados[label] += 1
            continue
        if contagem[label] < por_classe:
            exemplos.append({"text": texto, "label": label})
            contagem[label] += 1
        if all(v >= por_classe for v in contagem.values()):
            break
    random.Random(seed).shuffle(exemplos)
    print(split, {"coletados": contagem, "pulados": pulados})
    return exemplos


class SentimentDataset(Dataset):
    def __init__(self, exemplos: List[Dict], max_len: int = MAX_LEN):
        self.exemplos = exemplos
        self.max_len = max_len

    def __len__(self):
        return len(self.exemplos)

    def __getitem__(self, idx):
        ex = self.exemplos[idx]
        enc = codificar_texto(ex["text"], self.max_len)
        return {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "token_type_ids": torch.tensor(enc["token_type_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(ex["label"], dtype=torch.long),
            "text": ex["text"],
        }


sent_train = coletar_sentimentos("train", SENTIMENT_PER_CLASS_TRAIN, seed=SEED)

if SENTIMENT_VAL_MANUAL_EXAMPLES:
    sent_val = SENTIMENT_VAL_MANUAL_EXAMPLES
    print("validation manual", {"coletados": len(sent_val)})
else:
    sent_val = coletar_sentimentos(
        "validation",
        SENTIMENT_PER_CLASS_VAL,
        skip_por_classe=SENTIMENT_VAL_SKIP_PER_CLASS,
        seed=SEED + SENTIMENT_VAL_SKIP_PER_CLASS,
    )

sent_train_loader = DataLoader(SentimentDataset(sent_train), batch_size=DOWNSTREAM_BATCH_SIZE, shuffle=True, drop_last=True, **DL_KWARGS)
sent_val_loader = DataLoader(SentimentDataset(sent_val), batch_size=DOWNSTREAM_BATCH_SIZE, **DL_KWARGS)

batch = next(iter(sent_train_loader))
print("Sentiment input shape:", batch["input_ids"].shape)


Repo card metadata block was not found. Setting CardData to empty.


train {'coletados': {0: 1000, 1: 1000}, 'pulados': {0: 0, 1: 0}}


Repo card metadata block was not found. Setting CardData to empty.


validation {'coletados': {0: 200, 1: 200}, 'pulados': {0: 200, 1: 200}}
Sentiment input shape: torch.Size([8, 128])


In [76]:
def loss_sentimento(model, batch):
    logits = model(batch["input_ids"], batch["token_type_ids"], batch["attention_mask"])
    return F.cross_entropy(logits, batch["labels"])


@torch.no_grad()
def avaliar_sentimento(model):
    model.eval()
    y_true, y_pred = [], []
    for batch in sent_val_loader:
        batch = mover_batch(batch)
        logits = model(batch["input_ids"], batch["token_type_ids"], batch["attention_mask"])
        y_true.extend(batch["labels"].cpu().tolist())
        y_pred.extend(logits.argmax(dim=-1).cpu().tolist())
    print("Sentimento accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Sentimento F1:", round(f1_score(y_true, y_pred, zero_division=0), 4))
    model.train()


@torch.no_grad()
def prever_sentimento(model, textos: List[str]):
    model.eval()
    nomes = {0: "negativo", 1: "positivo"}
    for texto in textos:
        enc = codificar_texto(texto)
        batch = {k: torch.tensor([v], dtype=torch.long, device=DEVICE) for k, v in enc.items()}
        logits = model(batch["input_ids"], batch["token_type_ids"], batch["attention_mask"])
        probs = torch.softmax(logits[0], dim=-1).cpu().numpy()
        pred = int(probs.argmax())
        print(f"{nomes[pred]} | p={probs[pred]:.3f} | {texto[:160]}")
    model.train()


def textos_preview_sentimento(exemplos: List[Dict], indices: List[int] = SENTIMENT_PREVIEW_INDICES) -> List[str]:
    nomes = {0: "negativo", 1: "positivo"}
    textos = []
    for idx in indices:
        if 0 <= idx < len(exemplos):
            ex = exemplos[idx]
            print(f"Exemplo de validação {idx} | label real={nomes[int(ex['label'])]} | {ex['text'][:120]}")
            textos.append(ex["text"])
    return textos


sentiment_model = BertForSequenceClassificationDidatico(bert_config, num_labels=2)
copiar_encoder_pre_treinado(sentiment_model, pretrain_model)

treinar_por_epocas(
    "sentimentos",
    sentiment_model,
    sent_train_loader,
    avaliar_sentimento,
    loss_sentimento,
)

prever_sentimento(sentiment_model, textos_preview_sentimento(sent_val))


sentimentos epoch 1: 100%|██████████| 250/250 [00:10<00:00, 23.22it/s]


sentimentos epoch=1 loss=0.6423
Sentimento accuracy: 0.6775
Sentimento F1: 0.6861
Exemplo de validação 0 | label real=negativo | Unica coisa boa foi o visual, gostava quando saia alerta de gol, tabela demora atualizar, ficou ruim
Exemplo de validação 1 | label real=positivo | Muito bom ❤
Exemplo de validação 2 | label real=positivo | Viggo Mortensen lutando pelado numa sauna.Jamais esquecerei.
negativo | p=0.558 | Unica coisa boa foi o visual, gostava quando saia alerta de gol, tabela demora atualizar, ficou ruim
positivo | p=0.878 | Muito bom ❤
positivo | p=0.630 | Viggo Mortensen lutando pelado numa sauna.Jamais esquecerei.


# Tarefa B: NER

NER é uma tarefa de classificação por token. Como o WordPiece pode quebrar uma palavra em subpalavras, usamos a estratégia didática de treinar a label apenas no primeiro subtoken e usar `-100` nos demais, para que a loss ignore essas posições.


In [77]:
NER_LABELS = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
ID2NER = {i: label for i, label in enumerate(NER_LABELS)}
NER2ID = {label: i for i, label in ID2NER.items()}


def coletar_ner(split: str, n: int) -> List[Dict]:
    ds = load_dataset("unimelb-nlp/wikiann", "pt", split=split, streaming=True)
    exemplos = []
    for row in ds:
        tokens = row.get("tokens", [])
        tags = row.get("ner_tags", [])
        if tokens and len(tokens) == len(tags):
            exemplos.append({"tokens": tokens, "ner_tags": tags})
        if len(exemplos) >= n:
            break
    print(split, len(exemplos))
    return exemplos


class NERDataset(Dataset):
    def __init__(self, exemplos: List[Dict], max_len: int = MAX_LEN):
        self.exemplos = exemplos
        self.max_len = max_len

    def __len__(self):
        return len(self.exemplos)

    def __getitem__(self, idx):
        ex = self.exemplos[idx]
        enc = tokenizer.encode(ex["tokens"], is_pretokenized=True)
        ids = enc.ids[:self.max_len]
        type_ids = enc.type_ids[:self.max_len]
        word_ids = enc.word_ids[:self.max_len]

        labels = []
        prev_word = None
        for word_id in word_ids:
            if word_id is None:
                labels.append(-100)
            elif word_id != prev_word:
                labels.append(int(ex["ner_tags"][word_id]))
            else:
                labels.append(-100)
            prev_word = word_id

        if len(ids) == self.max_len and ids[-1] != SEP_ID:
            ids[-1] = SEP_ID
            labels[-1] = -100

        attention = [1] * len(ids)
        return {
            "input_ids": torch.tensor(pad_lista(ids, self.max_len, PAD_ID), dtype=torch.long),
            "token_type_ids": torch.tensor(pad_lista(type_ids, self.max_len, 0), dtype=torch.long),
            "attention_mask": torch.tensor(pad_lista(attention, self.max_len, 0), dtype=torch.long),
            "labels": torch.tensor(pad_lista(labels, self.max_len, -100), dtype=torch.long),
        }


ner_train = coletar_ner("train", NER_TRAIN_SIZE)
ner_val = coletar_ner("validation", NER_VAL_SIZE)
ner_train_loader = DataLoader(NERDataset(ner_train), batch_size=DOWNSTREAM_BATCH_SIZE, shuffle=True, drop_last=True, **DL_KWARGS)
ner_val_loader = DataLoader(NERDataset(ner_val), batch_size=DOWNSTREAM_BATCH_SIZE, **DL_KWARGS)

batch = next(iter(ner_train_loader))
print("NER input shape:", batch["input_ids"].shape)
print("NER labels shape:", batch["labels"].shape)


train 1000
validation 240
NER input shape: torch.Size([8, 128])
NER labels shape: torch.Size([8, 128])


In [78]:
def loss_ner(model, batch):
    logits = model(batch["input_ids"], batch["token_type_ids"], batch["attention_mask"])
    return F.cross_entropy(logits.view(-1, len(NER_LABELS)), batch["labels"].view(-1), ignore_index=-100)


@torch.no_grad()
def avaliar_ner(model):
    model.eval()
    y_true, y_pred = [], []
    for batch in ner_val_loader:
        batch = mover_batch(batch)
        logits = model(batch["input_ids"], batch["token_type_ids"], batch["attention_mask"])
        preds = logits.argmax(dim=-1).cpu().tolist()
        labels = batch["labels"].cpu().tolist()
        for pred_seq, label_seq in zip(preds, labels):
            seq_true, seq_pred = [], []
            for pred, label in zip(pred_seq, label_seq):
                if label != -100:
                    seq_true.append(ID2NER[int(label)])
                    seq_pred.append(ID2NER[int(pred)])
            if seq_true:
                y_true.append(seq_true)
                y_pred.append(seq_pred)
    print("NER F1 seqeval:", round(seqeval_f1(y_true, y_pred), 4))
    print(classification_report(y_true, y_pred, zero_division=0))
    model.train()


@torch.no_grad()
def prever_ner(model, tokens: List[str]):
    model.eval()
    ds = NERDataset([{"tokens": tokens, "ner_tags": [0] * len(tokens)}])
    batch = mover_batch(next(iter(DataLoader(ds, batch_size=1))))
    logits = model(batch["input_ids"], batch["token_type_ids"], batch["attention_mask"])
    preds = logits.argmax(dim=-1)[0].cpu().tolist()
    labels = []
    enc = tokenizer.encode(tokens, is_pretokenized=True)
    word_ids = enc.word_ids[:MAX_LEN]
    vistos = set()
    for pred, word_id in zip(preds, word_ids):
        if word_id is not None and word_id not in vistos:
            labels.append((tokens[word_id], ID2NER[pred]))
            vistos.add(word_id)
    print(labels)
    model.train()


ner_model = BertForTokenClassificationDidatico(bert_config, num_labels=len(NER_LABELS))
copiar_encoder_pre_treinado(ner_model, pretrain_model)

treinar_por_epocas(
    "ner",
    ner_model,
    ner_train_loader,
    avaliar_ner,
    loss_ner,
)

prever_ner(ner_model, ["Maria", "visitou", "São", "Paulo", "e", "trabalhou", "na", "Petrobras", "."])


ner epoch 1: 100%|██████████| 125/125 [00:04<00:00, 25.06it/s]


ner epoch=1 loss=1.5030
NER F1 seqeval: 0.0593
              precision    recall  f1-score   support

         LOC       0.00      0.00      0.00       119
         ORG       0.05      0.08      0.06        88
         PER       0.08      0.10      0.09       115

   micro avg       0.06      0.06      0.06       322
   macro avg       0.04      0.06      0.05       322
weighted avg       0.04      0.06      0.05       322

[('Maria', 'O'), ('visitou', 'O'), ('São', 'O'), ('Paulo', 'O'), ('e', 'O'), ('trabalhou', 'O'), ('na', 'O'), ('Petrobras', 'O'), ('.', 'O')]


# Tarefa C: Q&A extrativo

No Q&A extrativo, o modelo recebe pergunta e contexto. A resposta deve ser um trecho contíguo do contexto. Por isso, a cabeça da tarefa prevê duas posições: token inicial e token final.

O dataset português usado aqui é uma tradução do SQuAD. Como traduções automáticas podem deslocar spans, fazemos uma filtragem simples: só mantemos exemplos em que a resposta aparece de fato no contexto.


In [79]:
def buscar_resposta_no_contexto(contexto: str, resposta: str) -> Optional[Tuple[int, int]]:
    contexto = html.unescape(contexto or "")
    resposta = html.unescape(resposta or "").strip()
    if not contexto or not resposta:
        return None
    inicio = contexto.find(resposta)
    if inicio == -1:
        inicio_lower = contexto.lower().find(resposta.lower())
        if inicio_lower == -1:
            return None
        inicio = inicio_lower
    return inicio, inicio + len(resposta)


def extrair_primeira_resposta(row) -> Optional[Dict]:
    answers = row.get("answers") or {}
    textos = answers.get("text") or []
    if not textos:
        return None
    contexto = html.unescape(row.get("context", ""))
    pergunta = html.unescape(row.get("question", ""))
    resposta = html.unescape(textos[0])
    span = buscar_resposta_no_contexto(contexto, resposta)
    if span is None:
        return None
    return {
        "question": pergunta,
        "context": contexto,
        "answer_text": resposta,
        "answer_start": span[0],
        "answer_end": span[1],
    }


def coletar_qa(split: str, n: int) -> List[Dict]:
    ds = load_dataset("nunorc/squad_v1_pt", split=split, streaming=True)
    exemplos = []
    vistos = 0
    descartados = 0
    for row in ds:
        vistos += 1
        ex = extrair_primeira_resposta(row)
        if ex is None:
            descartados += 1
            continue
        exemplos.append(ex)
        if len(exemplos) >= n:
            break
    print(f"{split}: mantidos={len(exemplos)} descartados={descartados} lidos={vistos}")
    return exemplos


def criar_feature_qa(ex: Dict, max_len: int = MAX_LEN) -> Optional[Dict]:
    q_enc = tokenizer.encode(ex["question"], add_special_tokens=False)
    c_enc = tokenizer.encode(ex["context"], add_special_tokens=False)
    q_ids = q_enc.ids[:MAX_QUESTION_LEN]
    c_ids = c_enc.ids
    c_offsets = c_enc.offsets

    answer_start = ex["answer_start"]
    answer_end = ex["answer_end"]
    token_start = token_end = None
    for i, (ini, fim) in enumerate(c_offsets):
        if ini <= answer_start < fim:
            token_start = i
        if ini < answer_end <= fim:
            token_end = i
            break
    if token_start is None or token_end is None:
        return None

    max_context_len = max_len - len(q_ids) - 3
    if max_context_len <= 8:
        return None
    window_start = max(0, token_start - max_context_len // 3)
    window_end = min(len(c_ids), window_start + max_context_len)
    if token_end >= window_end:
        window_end = min(len(c_ids), token_end + 1)
        window_start = max(0, window_end - max_context_len)
    if not (window_start <= token_start < window_end and window_start <= token_end < window_end):
        return None

    prefix_len = 1 + len(q_ids) + 1
    ids = [CLS_ID] + q_ids + [SEP_ID] + c_ids[window_start:window_end] + [SEP_ID]
    type_ids = [0] * prefix_len + [1] * (window_end - window_start + 1)
    attention = [1] * len(ids)
    offsets = [(0, 0)] * prefix_len + c_offsets[window_start:window_end] + [(0, 0)]

    start_position = prefix_len + (token_start - window_start)
    end_position = prefix_len + (token_end - window_start)

    return {
        "input_ids": pad_lista(ids, max_len, PAD_ID),
        "token_type_ids": pad_lista(type_ids, max_len, 0),
        "attention_mask": pad_lista(attention, max_len, 0),
        "offsets": offsets + [(0, 0)] * max(0, max_len - len(offsets)),
        "start_positions": start_position,
        "end_positions": end_position,
        "question": ex["question"],
        "context": ex["context"],
        "answer_text": ex["answer_text"],
    }


class QADataset(Dataset):
    def __init__(self, exemplos: List[Dict], max_len: int = MAX_LEN):
        features = []
        for ex in exemplos:
            feat = criar_feature_qa(ex, max_len)
            if feat is not None:
                features.append(feat)
        self.features = features
        print("Features QA válidas:", len(self.features), "de", len(exemplos))

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        f = self.features[idx]
        return {
            "input_ids": torch.tensor(f["input_ids"], dtype=torch.long),
            "token_type_ids": torch.tensor(f["token_type_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(f["attention_mask"], dtype=torch.long),
            "offsets": torch.tensor(f["offsets"], dtype=torch.long),
            "start_positions": torch.tensor(f["start_positions"], dtype=torch.long),
            "end_positions": torch.tensor(f["end_positions"], dtype=torch.long),
            "question": f["question"],
            "context": f["context"],
            "answer_text": f["answer_text"],
        }


qa_train = coletar_qa("train", QA_TRAIN_SIZE)
qa_val = coletar_qa("validation", QA_VAL_SIZE)
qa_train_dataset = QADataset(qa_train)
qa_val_dataset = QADataset(qa_val)
qa_train_loader = DataLoader(qa_train_dataset, batch_size=max(2, DOWNSTREAM_BATCH_SIZE // 2), shuffle=True, drop_last=True, **DL_KWARGS)
qa_val_loader = DataLoader(qa_val_dataset, batch_size=max(2, DOWNSTREAM_BATCH_SIZE // 2), **DL_KWARGS)

batch = next(iter(qa_train_loader))
print("QA input shape:", batch["input_ids"].shape)
print("QA start/end:", batch["start_positions"][:3], batch["end_positions"][:3])


train: mantidos=350 descartados=152 lidos=502
validation: mantidos=80 descartados=27 lidos=107
Features QA válidas: 350 de 350
Features QA válidas: 80 de 80
QA input shape: torch.Size([4, 128])
QA start/end: tensor([53, 54, 54]) tensor([54, 54, 54])


In [80]:
def loss_qa(model, batch):
    start_logits, end_logits = model(batch["input_ids"], batch["token_type_ids"], batch["attention_mask"])
    start_loss = F.cross_entropy(start_logits, batch["start_positions"])
    end_loss = F.cross_entropy(end_logits, batch["end_positions"])
    return (start_loss + end_loss) / 2


def normalizar_para_metricas(texto: str) -> str:
    texto = texto.lower().strip()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(ch for ch in texto if unicodedata.category(ch) != "Mn")
    texto = re.sub(r"[^\w\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def f1_resposta(pred: str, gold: str) -> float:
    pred_tokens = normalizar_para_metricas(pred).split()
    gold_tokens = normalizar_para_metricas(gold).split()
    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)
    comum = set(pred_tokens) & set(gold_tokens)
    num_same = sum(min(pred_tokens.count(tok), gold_tokens.count(tok)) for tok in comum)
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def melhor_span(start_logits, end_logits, token_type_ids, attention_mask, max_answer_len: int = 30):
    contexto = [i for i, (t, m) in enumerate(zip(token_type_ids, attention_mask)) if int(t) == 1 and int(m) == 1]
    melhor = (0, 0, -1e30)
    for i in contexto:
        limite = min(i + max_answer_len, len(start_logits))
        for j in range(i, limite):
            if j not in contexto:
                continue
            score = float(start_logits[i] + end_logits[j])
            if score > melhor[2]:
                melhor = (i, j, score)
    return melhor[0], melhor[1]


@torch.no_grad()
def avaliar_qa(model):
    model.eval()
    ems, f1s = [], []
    exemplos_print = []
    for batch in qa_val_loader:
        batch_device = mover_batch(batch)
        start_logits, end_logits = model(
            batch_device["input_ids"],
            batch_device["token_type_ids"],
            batch_device["attention_mask"],
        )
        for i in range(start_logits.size(0)):
            s, e = melhor_span(
                start_logits[i].detach().cpu(),
                end_logits[i].detach().cpu(),
                batch["token_type_ids"][i].cpu(),
                batch["attention_mask"][i].cpu(),
            )
            offsets = batch["offsets"][i].tolist()
            ini, _ = offsets[s]
            _, fim = offsets[e]
            pred = batch["context"][i][ini:fim] if fim > ini else ""
            gold = batch["answer_text"][i]
            ems.append(float(normalizar_para_metricas(pred) == normalizar_para_metricas(gold)))
            f1s.append(f1_resposta(pred, gold))
            if len(exemplos_print) < 2:
                exemplos_print.append((batch["question"][i], pred, gold))
    print("QA EM:", round(float(np.mean(ems)), 4))
    print("QA F1:", round(float(np.mean(f1s)), 4))
    for pergunta, pred, gold in exemplos_print:
        print("Pergunta:", pergunta)
        print("Predição:", pred)
        print("Esperada:", gold)
        print("---")
    model.train()


qa_model = BertForQuestionAnsweringDidatico(bert_config)
copiar_encoder_pre_treinado(qa_model, pretrain_model)

treinar_por_epocas(
    "qa",
    qa_model,
    qa_train_loader,
    avaliar_qa,
    loss_qa,
)


qa epoch 1: 100%|██████████| 87/87 [00:02<00:00, 34.41it/s]


qa epoch=1 loss=4.3738
QA EM: 0.0625
QA F1: 0.1708
Pergunta: Qual time da NFL representou o AFC no Super Bowl 50?
Predição: 2015. O campeão da American Football Conference (AFC
Esperada: Denver Broncos
---
Pergunta: Qual time da NFL representou o NFC no Super Bowl 50?
Predição: 2015. O campeão da American Football Conference (AFC), Denver Broncos, derrotou a campeã Carolina Panther
Esperada: Carolina Panthers
---


In [81]:
@torch.no_grad()
def prever_qa(model, pergunta: str, contexto: str, stride: int = 64):
    model.eval()
    q_enc = tokenizer.encode(pergunta, add_special_tokens=False)
    c_enc = tokenizer.encode(contexto, add_special_tokens=False)
    q_ids = q_enc.ids[:MAX_QUESTION_LEN]
    c_ids = c_enc.ids
    c_offsets = c_enc.offsets
    max_context_len = MAX_LEN - len(q_ids) - 3
    passo = max(1, max_context_len - stride)

    melhores = []
    for window_start in range(0, len(c_ids), passo):
        window_end = min(len(c_ids), window_start + max_context_len)
        prefix_len = 1 + len(q_ids) + 1
        ids = [CLS_ID] + q_ids + [SEP_ID] + c_ids[window_start:window_end] + [SEP_ID]
        type_ids = [0] * prefix_len + [1] * (window_end - window_start + 1)
        attention = [1] * len(ids)
        offsets = [(0, 0)] * prefix_len + c_offsets[window_start:window_end] + [(0, 0)]

        batch = {
            "input_ids": torch.tensor([pad_lista(ids, MAX_LEN, PAD_ID)], dtype=torch.long, device=DEVICE),
            "token_type_ids": torch.tensor([pad_lista(type_ids, MAX_LEN, 0)], dtype=torch.long, device=DEVICE),
            "attention_mask": torch.tensor([pad_lista(attention, MAX_LEN, 0)], dtype=torch.long, device=DEVICE),
        }
        start_logits, end_logits = model(batch["input_ids"], batch["token_type_ids"], batch["attention_mask"])
        s, e = melhor_span(
            start_logits[0].cpu(),
            end_logits[0].cpu(),
            torch.tensor(pad_lista(type_ids, MAX_LEN, 0)),
            torch.tensor(pad_lista(attention, MAX_LEN, 0)),
        )
        score = float(start_logits[0, s].cpu() + end_logits[0, e].cpu())
        ini, _ = offsets[s]
        _, fim = offsets[e]
        resposta = contexto[ini:fim] if fim > ini else ""
        melhores.append((score, resposta))
        if window_end >= len(c_ids):
            break

    melhores.sort(reverse=True, key=lambda x: x[0])
    model.train()
    return melhores[0][1] if melhores else ""


if len(qa_val_dataset) > 0:
    exemplo = qa_val_dataset[0]
    resposta = prever_qa(qa_model, exemplo["question"], exemplo["context"])
    print("Pergunta:", exemplo["question"])
    print("Resposta prevista:", resposta)
    print("Resposta esperada:", exemplo["answer_text"])


Pergunta: Qual time da NFL representou o AFC no Super Bowl 50?
Resposta prevista: 2016
Resposta esperada: Denver Broncos


## Conclusões

Neste notebook, construímos um BERT pequeno do zero e deixamos o fluxo preparado para execução no Google Colab, mantendo o registro do treino longo realizado no iMac M4:

- o tokenizer WordPiece foi treinado em corpus real em português e é recarregado do arquivo salvo quando já existe;
- os pesos do BERT começam aleatórios, ou seja, o modelo é realmente treinado do zero;
- a Wikipedia em português fica disponível como fonte de pré-treinamento em streaming, sem carregar todos os artigos na RAM;
- o pré-treinamento usa MLM e NSP, com checkpoints automáticos;
- o checkpoint principal usado no trabalho foi treinado localmente no iMac M4 com MPS;
- no Colab, o preset padrão é mais leve e serve para demonstração, inferência, retomada controlada ou fine-tuning downstream;
- o fine-tuning reutiliza o encoder pré-treinado em sentimentos, NER e Q&A extrativo.

Na configuração original do iMac, `MAX_WIKI_DOCS` apontava para cerca de 1,1 milhão de artigos, `MAX_SENTENCES_PER_DOC = 8`, `PRETRAIN_BATCH_SIZE = 32` e `PRETRAIN_STEPS = 220_000`. Esses valores foram mantidos comentados na célula de configuração. No Colab, usamos um preset menor para reduzir risco de estouro de memória e facilitar a execução do notebook.

Para melhorar os resultados, o caminho natural é:

1. carregar no Colab o tokenizer e o checkpoint gerados no iMac;
2. continuar o pré-treinamento a partir do último checkpoint, em vez de recomeçar do zero;
3. acompanhar `mlm_loss`, `nsp_loss`, `mlm_acc` e `nsp_acc`;
4. treinar por mais épocas nas tarefas downstream;
5. manter checkpoints frequentes para poder interromper e retomar o treinamento com segurança.
